# Medicare Provider Analytics Using Apache Spark

## Data Cleaning and Preparation

**Course:** CS-675 Big Data Management & Analytics  
**Author:** Judi-Ann Beckford

### Objective

Clean and prepare the Medicare Provider Service and Public Provider Enrollment datasets for integration using the National Provider Identifier (NPI).

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

In [4]:
spark = (
    SparkSession.builder
    .appName("Medicare Data Cleaning")
    .config("spark.sql.shuffle.partitions", "24")
    .config("spark.default.parallelism", "24")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Driver memory:", spark.sparkContext.getConf().get("spark.driver.memory"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
26/08/31 20:43:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2
Spark master: local[*]
Driver memory: None


In [5]:
provider_file = "../data/raw/PHY_R26_P05_V10_D24_Prov_Svc.csv"
enrollment_file = "../data/raw/PPEF_Enrollment_Extract_2026.04.01.csv"

print("Provider file exists:", os.path.exists(provider_file))
print("Enrollment file exists:", os.path.exists(enrollment_file))
print("Current folder:", os.getcwd())

Provider file exists: True
Enrollment file exists: True
Current folder: /Users/judi-annbeckford/Documents/Medicare-Provider-Analytics-Spark/notebooks


In [6]:
provider_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(provider_file)
)

enrollment_raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(enrollment_file)
)

print("Both raw datasets loaded.")

[Stage 3:====================================================>    (22 + 2) / 24]

Both raw datasets loaded.


In [8]:
provider_clean_df = (
    provider_raw_df
    .select(
        F.col("Rndrng_NPI").cast("long").alias("NPI"),
        F.trim(F.col("Rndrng_Prvdr_Type")).alias("PROVIDER_SPECIALTY"),
        F.upper(F.trim(F.col("Rndrng_Prvdr_State_Abrvtn"))).alias("PROVIDER_STATE"),
        F.trim(F.col("HCPCS_Cd")).alias("HCPCS_CODE"),
        F.trim(F.col("HCPCS_Desc")).alias("HCPCS_DESCRIPTION"),
        F.trim(F.col("Place_Of_Srvc")).alias("PLACE_OF_SERVICE"),
        F.col("Tot_Benes").cast("double").alias("TOTAL_BENEFICIARIES"),
        F.col("Tot_Srvcs").cast("double").alias("TOTAL_SERVICES"),
        F.col("Avg_Sbmtd_Chrg").cast("double").alias("AVG_SUBMITTED_CHARGE"),
        F.col("Avg_Mdcr_Alowd_Amt").cast("double").alias("AVG_MEDICARE_ALLOWED"),
        F.col("Avg_Mdcr_Pymt_Amt").cast("double").alias("AVG_MEDICARE_PAYMENT"),
        F.col("Avg_Mdcr_Stdzd_Amt").cast("double").alias("AVG_STANDARDIZED_PAYMENT")
    )
    .filter(F.col("NPI").isNotNull())
    .filter(F.col("HCPCS_CODE").isNotNull())
)

In [9]:
enrollment_clean_df = (
    enrollment_raw_df
    .select(
        F.col("NPI").cast("long").alias("NPI"),
        F.trim(F.col("PROVIDER_TYPE_DESC")).alias("ENROLLMENT_PROVIDER_TYPE"),
        F.upper(F.trim(F.col("STATE_CD"))).alias("ENROLLMENT_STATE"),
        F.trim(F.col("FIRST_NAME")).alias("FIRST_NAME"),
        F.trim(F.col("MDL_NAME")).alias("MIDDLE_NAME"),
        F.trim(F.col("LAST_NAME")).alias("LAST_NAME"),
        F.trim(F.col("ORG_NAME")).alias("ORGANIZATION_NAME")
    )
    .filter(F.col("NPI").isNotNull())
)

In [10]:
print("CLEAN PROVIDER-SERVICE SCHEMA")
provider_clean_df.printSchema()

print("\nCLEAN ENROLLMENT SCHEMA")
enrollment_clean_df.printSchema()

CLEAN PROVIDER-SERVICE SCHEMA
root
 |-- NPI: long (nullable = true)
 |-- PROVIDER_SPECIALTY: string (nullable = true)
 |-- PROVIDER_STATE: string (nullable = true)
 |-- HCPCS_CODE: string (nullable = true)
 |-- HCPCS_DESCRIPTION: string (nullable = true)
 |-- PLACE_OF_SERVICE: string (nullable = true)
 |-- TOTAL_BENEFICIARIES: double (nullable = true)
 |-- TOTAL_SERVICES: double (nullable = true)
 |-- AVG_SUBMITTED_CHARGE: double (nullable = true)
 |-- AVG_MEDICARE_ALLOWED: double (nullable = true)
 |-- AVG_MEDICARE_PAYMENT: double (nullable = true)
 |-- AVG_STANDARDIZED_PAYMENT: double (nullable = true)


CLEAN ENROLLMENT SCHEMA
root
 |-- NPI: long (nullable = true)
 |-- ENROLLMENT_PROVIDER_TYPE: string (nullable = true)
 |-- ENROLLMENT_STATE: string (nullable = true)
 |-- FIRST_NAME: string (nullable = true)
 |-- MIDDLE_NAME: string (nullable = true)
 |-- LAST_NAME: string (nullable = true)
 |-- ORGANIZATION_NAME: string (nullable = true)



In [11]:
provider_clean_df.show(5, truncate=False)

+----------+------------------+--------------+----------+----------------------------------------------------------------------------------------------------------------------------------+----------------+-------------------+--------------+--------------------+--------------------+--------------------+------------------------+
|NPI       |PROVIDER_SPECIALTY|PROVIDER_STATE|HCPCS_CODE|HCPCS_DESCRIPTION                                                                                                                 |PLACE_OF_SERVICE|TOTAL_BENEFICIARIES|TOTAL_SERVICES|AVG_SUBMITTED_CHARGE|AVG_MEDICARE_ALLOWED|AVG_MEDICARE_PAYMENT|AVG_STANDARDIZED_PAYMENT|
+----------+------------------+--------------+----------+----------------------------------------------------------------------------------------------------------------------------------+----------------+-------------------+--------------+--------------------+--------------------+--------------------+------------------------+
|1003000126|I

In [12]:
enrollment_clean_df.show(5, truncate=False)

+----------+--------------------------------+----------------+----------+-----------+---------+-----------------+
|NPI       |ENROLLMENT_PROVIDER_TYPE        |ENROLLMENT_STATE|FIRST_NAME|MIDDLE_NAME|LAST_NAME|ORGANIZATION_NAME|
+----------+--------------------------------+----------------+----------+-----------+---------+-----------------+
|1003000126|PRACTITIONER - INTERNAL MEDICINE|MD              |ARDALAN   |NULL       |ENKESHAFI|NULL             |
|1003000126|PRACTITIONER - HOSPITALIST      |DC              |ARDALAN   |NULL       |ENKESHAFI|NULL             |
|1003000126|PRACTITIONER - INTERNAL MEDICINE|VA              |ARDALAN   |NULL       |ENKESHAFI|NULL             |
|1003000126|PRACTITIONER - INTERNAL MEDICINE|PA              |ARDALAN   |NULL       |ENKESHAFI|NULL             |
|1003000134|PRACTITIONER - PATHOLOGY        |IL              |THOMAS    |L          |CIBULL   |NULL             |
+----------+--------------------------------+----------------+----------+-----------+---

Cleaning Completed So Far

- Selected only the variables required for the analytical questions.
- Standardized both NPI fields as long integers.
- Standardized state abbreviations using uppercase values.
- Trimmed unnecessary whitespace from categorical variables.
- Converted service, beneficiary, charge, and payment variables to numeric types.
- Removed records without an NPI.
- Removed provider-service records without an HCPCS procedure code.
- Retained enrollment duplicates temporarily to prevent accidental loss of legitimate provider records.

In [14]:
# Preprocessing assessment: missing values and numeric distributions

numeric_cols = [
    "TOTAL_BENEFICIARIES",
    "TOTAL_SERVICES",
    "AVG_SUBMITTED_CHARGE",
    "AVG_MEDICARE_ALLOWED",
    "AVG_MEDICARE_PAYMENT",
    "AVG_STANDARDIZED_PAYMENT"
]

print("=== MISSING VALUES IN CLEAN PROVIDER-SERVICE DATA ===")

provider_clean_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in provider_clean_df.columns
]).show(truncate=False)

print("=== NUMERIC DISTRIBUTION SUMMARY ===")

provider_clean_df.select(numeric_cols).summary(
    "count", "min", "25%", "50%", "75%", "max", "mean", "stddev"
).show(truncate=False)

=== MISSING VALUES IN CLEAN PROVIDER-SERVICE DATA ===


+---+------------------+--------------+----------+-----------------+----------------+-------------------+--------------+--------------------+--------------------+--------------------+------------------------+
|NPI|PROVIDER_SPECIALTY|PROVIDER_STATE|HCPCS_CODE|HCPCS_DESCRIPTION|PLACE_OF_SERVICE|TOTAL_BENEFICIARIES|TOTAL_SERVICES|AVG_SUBMITTED_CHARGE|AVG_MEDICARE_ALLOWED|AVG_MEDICARE_PAYMENT|AVG_STANDARDIZED_PAYMENT|
+---+------------------+--------------+----------+-----------------+----------------+-------------------+--------------+--------------------+--------------------+--------------------+------------------------+
|0  |0                 |0             |0         |0                |0               |0                  |0             |0                   |0                   |0                   |0                       |
+---+------------------+--------------+----------+-----------------+----------------+-------------------+--------------+--------------------+--------------------+--

[Stage 11:>                                                         (0 + 1) / 1]

+-------+-------------------+------------------+--------------------+--------------------+--------------------+------------------------+
|summary|TOTAL_BENEFICIARIES|TOTAL_SERVICES    |AVG_SUBMITTED_CHARGE|AVG_MEDICARE_ALLOWED|AVG_MEDICARE_PAYMENT|AVG_STANDARDIZED_PAYMENT|
+-------+-------------------+------------------+--------------------+--------------------+--------------------+------------------------+
|count  |9781673            |9781673           |9781673             |9781673             |9781673             |9781673                 |
|min    |11.0               |4.4               |3.50877E-5          |0.0                 |0.0                 |0.0                     |
|25%    |17.0               |21.0              |73.256036036        |25.481948052        |19.648055556        |19.468                  |
|50%    |32.0               |43.0              |180.0               |71.549647059        |52.453141593        |52.648913043            |
|75%    |72.0               |119.0       

Outlier Detection and Treatment

The numeric distribution analysis shows substantial right-skew in Medicare utilization and payment variables. Extreme values may represent legitimate high-volume providers or expensive services rather than data-entry errors. Therefore, observations will not be deleted solely because they are statistical outliers.

For analytical features, the Interquartile Range (IQR) method is used to identify extreme values. Capped versions of selected variables will be created where appropriate while preserving the original Medicare values for reporting and interpretation.

In [16]:
# Detect outliers using the IQR method

outlier_cols = [
    "TOTAL_BENEFICIARIES",
    "TOTAL_SERVICES",
    "AVG_SUBMITTED_CHARGE",
    "AVG_MEDICARE_ALLOWED",
    "AVG_MEDICARE_PAYMENT",
    "AVG_STANDARDIZED_PAYMENT"
]

outlier_bounds = {}

for c in outlier_cols:
    q1, q3 = provider_clean_df.approxQuantile(c, [0.25, 0.75], 0.01)
    iqr = q3 - q1

    lower_bound = max(0.0, q1 - 1.5 * iqr)
    upper_bound = q3 + 1.5 * iqr

    outlier_bounds[c] = (lower_bound, upper_bound)

    outlier_count = provider_clean_df.filter(
        (F.col(c) < lower_bound) |
        (F.col(c) > upper_bound)
    ).count()

    print(
        f"{c}: "
        f"Q1={q1:.2f}, "
        f"Q3={q3:.2f}, "
        f"IQR={iqr:.2f}, "
        f"Lower={lower_bound:.2f}, "
        f"Upper={upper_bound:.2f}, "
        f"Outliers={outlier_count:,}"
    )

TOTAL_BENEFICIARIES: Q1=18.00, Q3=71.00, IQR=53.00, Lower=0.00, Upper=150.50, Outliers=1,040,322


TOTAL_SERVICES: Q1=21.00, Q3=117.00, IQR=96.00, Lower=0.00, Upper=261.00, Outliers=1,214,307


AVG_SUBMITTED_CHARGE: Q1=72.46, Q3=362.00, IQR=289.54, Lower=0.00, Upper=796.31, Outliers=1,013,551


AVG_MEDICARE_ALLOWED: Q1=25.12, Q3=120.42, IQR=95.29, Lower=0.00, Upper=263.36, Outliers=466,050


AVG_MEDICARE_PAYMENT: Q1=19.47, Q3=90.35, IQR=70.88, Lower=0.00, Upper=196.66, Outliers=498,376


[Stage 45:===================================>                    (16 + 8) / 25]

AVG_STANDARDIZED_PAYMENT: Q1=19.29, Q3=89.95, IQR=70.66, Lower=0.00, Upper=195.94, Outliers=490,861


Outlier Treatment

IQR analysis identified substantial right-tail outliers across utilization and payment measures. These records were not deleted because unusually high Medicare utilization or payment values may represent legitimate provider activity.

Instead, capped analytical versions of the numeric variables were created using the IQR upper bounds. The original values are retained for reporting and business interpretation, while the capped features provide less skewed inputs for comparative analysis and subsequent transformations.

In [18]:
# Create capped analytical features while preserving original values

provider_preprocessed_df = provider_clean_df

for c, (lower_bound, upper_bound) in outlier_bounds.items():
    provider_preprocessed_df = provider_preprocessed_df.withColumn(
        f"{c}_CAPPED",
        F.when(F.col(c) < lower_bound, F.lit(lower_bound))
         .when(F.col(c) > upper_bound, F.lit(upper_bound))
         .otherwise(F.col(c))
    )

print("Original and capped values created successfully.")

provider_preprocessed_df.select(
    "TOTAL_SERVICES",
    "TOTAL_SERVICES_CAPPED",
    "AVG_MEDICARE_PAYMENT",
    "AVG_MEDICARE_PAYMENT_CAPPED"
).orderBy(F.col("TOTAL_SERVICES").desc()).show(10, truncate=False)

Original and capped values created successfully.


[Stage 48:===================================>                    (16 + 8) / 25]

+--------------+---------------------+--------------------+---------------------------+
|TOTAL_SERVICES|TOTAL_SERVICES_CAPPED|AVG_MEDICARE_PAYMENT|AVG_MEDICARE_PAYMENT_CAPPED|
+--------------+---------------------+--------------------+---------------------------+
|6486401.0     |261.0                |0.5079266977        |0.5079266977               |
|4461619.0     |261.0                |0.5130369155        |0.5130369155               |
|4371834.0     |261.0                |1.063943448         |1.063943448                |
|4265051.0     |261.0                |1.2007528585        |1.2007528585               |
|4075672.0     |261.0                |1.1981462861        |1.1981462861               |
|3498636.0     |261.0                |1.0746666329        |1.0746666329               |
|3232019.0     |261.0                |1.078125735         |1.078125735                |
|2728340.0     |261.0                |1.0983888262        |1.0983888262               |
|2637392.8     |261.0           

In [20]:
# Min-Max normalization of selected capped analytical features

normalization_cols = [
    "TOTAL_BENEFICIARIES_CAPPED",
    "TOTAL_SERVICES_CAPPED",
    "AVG_SUBMITTED_CHARGE_CAPPED",
    "AVG_MEDICARE_ALLOWED_CAPPED",
    "AVG_MEDICARE_PAYMENT_CAPPED",
    "AVG_STANDARDIZED_PAYMENT_CAPPED"
]

for c in normalization_cols:
    stats = provider_preprocessed_df.agg(
        F.min(c).alias("min_value"),
        F.max(c).alias("max_value")
    ).first()

    min_value = stats["min_value"]
    max_value = stats["max_value"]

    provider_preprocessed_df = provider_preprocessed_df.withColumn(
        f"{c}_NORM",
        F.when(
            F.lit(max_value) != F.lit(min_value),
            (F.col(c) - F.lit(min_value)) /
            (F.lit(max_value) - F.lit(min_value))
        ).otherwise(F.lit(0.0))
    )

print("Min-Max normalization completed.")

provider_preprocessed_df.select(
    "TOTAL_SERVICES",
    "TOTAL_SERVICES_CAPPED",
    "TOTAL_SERVICES_CAPPED_NORM",
    "AVG_MEDICARE_PAYMENT",
    "AVG_MEDICARE_PAYMENT_CAPPED",
    "AVG_MEDICARE_PAYMENT_CAPPED_NORM"
).show(10, truncate=False)

[Stage 64:===================================>                    (16 + 8) / 25]

Min-Max normalization completed.
+--------------+---------------------+--------------------------+--------------------+---------------------------+--------------------------------+
|TOTAL_SERVICES|TOTAL_SERVICES_CAPPED|TOTAL_SERVICES_CAPPED_NORM|AVG_MEDICARE_PAYMENT|AVG_MEDICARE_PAYMENT_CAPPED|AVG_MEDICARE_PAYMENT_CAPPED_NORM|
+--------------+---------------------+--------------------------+--------------------+---------------------------+--------------------------------+
|36.0          |36.0                 |0.12314886983632112       |60.828888889        |60.828888889               |0.30930256175635273             |
|150.0         |150.0                |0.5674201091192517        |95.6756             |95.6756                    |0.4864910196136676              |
|63.0          |63.0                 |0.22837100545596256       |134.30047619        |134.30047619               |0.6828906805525566              |
|16.0          |16.0                 |0.04520654715510522       |33.350625     

Categorical Encoding

Categorical variables are useful for grouping and interpretation but cannot always be used directly by analytical or machine-learning algorithms.

The `PLACE_OF_SERVICE` field is therefore encoded using Spark's `StringIndexer`. The original categorical value is retained for business interpretation, while the indexed version provides a numeric analytical representation.

Encoding is demonstrated as part of the preprocessing pipeline without replacing the original Medicare category.

In [21]:
from pyspark.ml.feature import StringIndexer

# Encode PLACE_OF_SERVICE while preserving the original category

place_indexer = StringIndexer(
    inputCol="PLACE_OF_SERVICE",
    outputCol="PLACE_OF_SERVICE_INDEX",
    handleInvalid="keep"
)

place_indexer_model = place_indexer.fit(provider_preprocessed_df)

provider_preprocessed_df = place_indexer_model.transform(
    provider_preprocessed_df
)

print("Categorical encoding completed.")

provider_preprocessed_df.select(
    "PLACE_OF_SERVICE",
    "PLACE_OF_SERVICE_INDEX"
).distinct().orderBy(
    "PLACE_OF_SERVICE_INDEX"
).show(truncate=False)

Categorical encoding completed.


[Stage 74:===================================>                    (16 + 8) / 25]

+----------------+----------------------+
|PLACE_OF_SERVICE|PLACE_OF_SERVICE_INDEX|
+----------------+----------------------+
|O               |0.0                   |
|F               |1.0                   |
+----------------+----------------------+



Utilization Binning

To make Medicare service utilization easier to interpret, `TOTAL_SERVICES` is converted into categorical utilization bands.

The bin boundaries are based on the observed quartiles of the provider-service dataset rather than arbitrary thresholds:

- Low Utilization: 21 services or fewer
- Moderate Utilization: 22–43 services
- High Utilization: 44–117 services
- Very High Utilization: more than 117 services

The original numeric service count is retained while the binned feature provides a simpler categorical representation for comparison and reporting.

In [22]:
# Create utilization bands using observed quartiles

provider_preprocessed_df = provider_preprocessed_df.withColumn(
    "UTILIZATION_BAND",
    F.when(F.col("TOTAL_SERVICES") <= 21, "Low")
     .when(F.col("TOTAL_SERVICES") <= 43, "Moderate")
     .when(F.col("TOTAL_SERVICES") <= 117, "High")
     .otherwise("Very High")
)

print("Utilization binning completed.")

provider_preprocessed_df.groupBy(
    "UTILIZATION_BAND"
).count().orderBy(
    F.when(F.col("UTILIZATION_BAND") == "Low", 1)
     .when(F.col("UTILIZATION_BAND") == "Moderate", 2)
     .when(F.col("UTILIZATION_BAND") == "High", 3)
     .otherwise(4)
).show(truncate=False)

Utilization binning completed.


[Stage 77:===================================>                    (16 + 8) / 25]

+----------------+-------+
|UTILIZATION_BAND|count  |
+----------------+-------+
|Low             |2588720|
|Moderate        |2303072|
|High            |2423812|
|Very High       |2466069|
+----------------+-------+



Preprocessing Validation and Summary

The preprocessing pipeline was validated after transformation to confirm that data quality was improved without unnecessarily removing legitimate Medicare activity.

Preprocessing Decisions

- Missing values: Missingness was evaluated after initial cleaning. No null values remained in the selected provider-service analytical fields, so artificial imputation was not required. Records missing critical identifiers such as NPI or HCPCS code were excluded because those values cannot be meaningfully imputed.
- Outliers: IQR analysis identified substantial right-tail outliers. Original observations were retained because extreme Medicare utilization may represent legitimate provider activity. Capped analytical features were created instead of deleting these records.
- Normalization: Selected capped numeric features were transformed to a 0–1 scale using Min-Max normalization while original values were preserved.
- Encoding: `PLACE_OF_SERVICE` was converted to a numeric analytical feature using Spark StringIndexer while retaining the original categorical value.
- Binning: `TOTAL_SERVICES` was categorized into utilization bands using quartile-based thresholds derived from the dataset.

These transformations preserve the original Medicare measures while creating additional features suitable for large-scale analytical processing.

In [23]:
# Final preprocessing validation

original_count = provider_clean_df.count()
preprocessed_count = provider_preprocessed_df.count()

print("=== PREPROCESSING VALIDATION ===")
print(f"Original cleaned rows:     {original_count:,}")
print(f"Preprocessed rows:         {preprocessed_count:,}")
print(f"Rows lost during feature engineering: {original_count - preprocessed_count:,}")

print("\n=== NORMALIZATION RANGE CHECK ===")

provider_preprocessed_df.select(
    F.min("TOTAL_SERVICES_CAPPED_NORM").alias("SERVICES_NORM_MIN"),
    F.max("TOTAL_SERVICES_CAPPED_NORM").alias("SERVICES_NORM_MAX"),
    F.min("AVG_MEDICARE_PAYMENT_CAPPED_NORM").alias("PAYMENT_NORM_MIN"),
    F.max("AVG_MEDICARE_PAYMENT_CAPPED_NORM").alias("PAYMENT_NORM_MAX")
).show(truncate=False)

print("\n=== FEATURE ENGINEERING SAMPLE ===")

provider_preprocessed_df.select(
    "NPI",
    "TOTAL_SERVICES",
    "TOTAL_SERVICES_CAPPED",
    "TOTAL_SERVICES_CAPPED_NORM",
    "UTILIZATION_BAND",
    "PLACE_OF_SERVICE",
    "PLACE_OF_SERVICE_INDEX",
    "AVG_MEDICARE_PAYMENT",
    "AVG_MEDICARE_PAYMENT_CAPPED",
    "AVG_MEDICARE_PAYMENT_CAPPED_NORM"
).show(10, truncate=False)

=== PREPROCESSING VALIDATION ===
Original cleaned rows:     9,781,673
Preprocessed rows:         9,781,673
Rows lost during feature engineering: 0

=== NORMALIZATION RANGE CHECK ===


[Stage 86:===================================>                    (16 + 8) / 25]

+-----------------+-----------------+----------------+----------------+
|SERVICES_NORM_MIN|SERVICES_NORM_MAX|PAYMENT_NORM_MIN|PAYMENT_NORM_MAX|
+-----------------+-----------------+----------------+----------------+
|0.0              |1.0              |0.0             |1.0             |
+-----------------+-----------------+----------------+----------------+


=== FEATURE ENGINEERING SAMPLE ===
+----------+--------------+---------------------+--------------------------+----------------+----------------+----------------------+--------------------+---------------------------+--------------------------------+
|NPI       |TOTAL_SERVICES|TOTAL_SERVICES_CAPPED|TOTAL_SERVICES_CAPPED_NORM|UTILIZATION_BAND|PLACE_OF_SERVICE|PLACE_OF_SERVICE_INDEX|AVG_MEDICARE_PAYMENT|AVG_MEDICARE_PAYMENT_CAPPED|AVG_MEDICARE_PAYMENT_CAPPED_NORM|
+----------+--------------+---------------------+--------------------------+----------------+----------------+----------------------+--------------------+-----------------

Save Preprocessed Dataset

The completed provider-service preprocessing output is stored in Parquet format for downstream integration and analytics.

Parquet was selected because it provides columnar storage, schema preservation, compression, and efficient column pruning in Apache Spark. The output is partitioned by provider state to support more efficient state-level analytical queries.

The raw CMS source files remain unchanged.


In [24]:
# Save enhanced preprocessing output for downstream integration

preprocessed_output = "../data/processed/provider_services_preprocessed"

(
    provider_preprocessed_df
    .write
    .mode("overwrite")
    .partitionBy("PROVIDER_STATE")
    .parquet(preprocessed_output)
)

print("Preprocessed provider-service dataset saved successfully.")
print("Output:", preprocessed_output)

Preprocessed provider-service dataset saved successfully.
Output: ../data/processed/provider_services_preprocessed


In [25]:
# Verify saved preprocessing output

preprocessed_check_df = spark.read.parquet(preprocessed_output)

print(f"Saved row count: {preprocessed_check_df.count():,}")
print(f"Saved column count: {len(preprocessed_check_df.columns)}")

print("\nSaved feature columns:")
for c in preprocessed_check_df.columns:
    print(c)

Saved row count: 9,781,673
Saved column count: 26

Saved feature columns:
NPI
PROVIDER_SPECIALTY
HCPCS_CODE
HCPCS_DESCRIPTION
PLACE_OF_SERVICE
TOTAL_BENEFICIARIES
TOTAL_SERVICES
AVG_SUBMITTED_CHARGE
AVG_MEDICARE_ALLOWED
AVG_MEDICARE_PAYMENT
AVG_STANDARDIZED_PAYMENT
TOTAL_BENEFICIARIES_CAPPED
TOTAL_SERVICES_CAPPED
AVG_SUBMITTED_CHARGE_CAPPED
AVG_MEDICARE_ALLOWED_CAPPED
AVG_MEDICARE_PAYMENT_CAPPED
AVG_STANDARDIZED_PAYMENT_CAPPED
TOTAL_BENEFICIARIES_CAPPED_NORM
TOTAL_SERVICES_CAPPED_NORM
AVG_SUBMITTED_CHARGE_CAPPED_NORM
AVG_MEDICARE_ALLOWED_CAPPED_NORM
AVG_MEDICARE_PAYMENT_CAPPED_NORM
AVG_STANDARDIZED_PAYMENT_CAPPED_NORM
PLACE_OF_SERVICE_INDEX
UTILIZATION_BAND
PROVIDER_STATE
